In [ ]:
#解壓縮資料集，取得 Training Images、Ground Truth Masks 與 Test Images，供後續模型訓練、驗證與預測使用
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount("/content/drive")

# 自動搜尋 Google Drive 裡的作業資料集
zip_files = list(
    Path("/content/drive/MyDrive").rglob(
        "*ntnuee-115-1-dl-homework-1*.zip"
    )
)

print("找到的 ZIP：", zip_files)

assert len(zip_files) == 1, (
    f"預期找到 1 個 ZIP，實際找到 {len(zip_files)} 個，"
    "請確認檔案名稱或位置。"
)

ZIP_PATH = zip_files[0]

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall("/content")

print("解壓縮完成：", ZIP_PATH)

In [ ]:
# 確認 Mask 跟 Image 有對上
from pathlib import Path
from PIL import Image
import numpy as np

# 定義資料夾路徑
TRAIN_IMAGE_DIR = Path("/content/DL_HW1_Public/train/image")
TRAIN_MASK_DIR = Path("/content/DL_HW1_Public/train/gt")

VAL_IMAGE_DIR = Path("/content/DL_HW1_Public/val/image")
VAL_MASK_DIR = Path("/content/DL_HW1_Public/val/mask")

# Test 路徑：如果前面已經設定 TEST_IMAGE_DIR，就沿用原本設定
# 如果尚未設定，請先確認實際的測試圖片資料夾位置
if "TEST_IMAGE_DIR" not in globals():
    candidates = [
        Path("/content/DL_HW1_Public/test/image"),
        Path("/content/DL_HW1_Public/test/images"),
    ]
    existing = [p for p in candidates if p.is_dir()]

    if len(existing) == 1:
        TEST_IMAGE_DIR = existing[0]
    else:
        raise FileNotFoundError(
            "無法自動確認 Test 圖片資料夾，"
            "請先設定 TEST_IMAGE_DIR 為實際路徑。"
        )

TEST_IMAGE_DIR = Path(TEST_IMAGE_DIR)

# 確認資料夾存在
assert TRAIN_IMAGE_DIR.is_dir(), f"找不到訓練圖片資料夾：{TRAIN_IMAGE_DIR}"
assert TRAIN_MASK_DIR.is_dir(), f"找不到訓練標註資料夾：{TRAIN_MASK_DIR}"
assert TEST_IMAGE_DIR.is_dir(), f"找不到測試圖片資料夾：{TEST_IMAGE_DIR}"

# 定義列出資料夾所有合法圖片檔的函式
def list_images(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )

# 分別取得各資料集檔案列表
train_images = list_images(TRAIN_IMAGE_DIR)
train_masks = list_images(TRAIN_MASK_DIR)
test_images = list_images(TEST_IMAGE_DIR)

# 建立 Mask 檔名對應字典
mask_by_stem = {p.stem: p for p in train_masks}

# 檢查是否有訓練圖片缺少對應 Mask
missing_masks = [
    p.name for p in train_images
    if p.stem not in mask_by_stem
]

# 印出資料數量
print("Train images:", len(train_images))
print("Train masks:", len(train_masks))
print("Test images:", len(test_images))
print("Missing masks:", len(missing_masks))

# 確認資料數量
assert len(train_images) == 5000, "訓練圖片數量不是 5000"
assert len(train_masks) == 5000, "訓練標註數量不是 5000"
assert len(test_images) == 1000, "測試圖片數量不是 1000"

# 確認圖片和 Mask 一一對應
if missing_masks:
    raise ValueError(
        f"有 {len(missing_masks)} 張圖片缺少對應標註："
        f"{missing_masks[:10]}"
    )

# 按照 Image 順序排列 Mask，確保一一對應
train_masks = [
    mask_by_stem[p.stem]
    for p in train_images
]

print("Image / Mask 配對檢查完成")

In [ ]:
# 切分 Training / Validation
import random
import shutil
from pathlib import Path

# 使用固定 Random Seed = 42
# 將原始 5000 張資料切分為 4500 張 Training 和 500 張 Validation
SEED = 42

# 自動建立驗證集資料夾
VAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
VAL_MASK_DIR.mkdir(parents=True, exist_ok=True)

# 確認原始資料數量
assert len(train_images) == 5000
assert len(train_masks) == 5000

# 固定隨機切分，建立 0~4999 索引列表後隨機打散
indices = list(range(len(train_images)))
random.Random(SEED).shuffle(indices)

val_indices = indices[:500]     # 前 500 張作為 Validation
train_indices = indices[500:]   # 後 4500 張作為 Training

# 確認圖片與 Mask 一一對應
for i in range(len(train_images)):
    assert train_images[i].stem == train_masks[i].stem, (
        f"Image-mask mismatch: "
        f"{train_images[i]} vs {train_masks[i]}"
    )

# 複製 500 張 Validation 圖片與 Mask 到 val 資料夾
for idx in val_indices:
    image_path = train_images[idx]
    mask_path = train_masks[idx]

    shutil.copy2(
        image_path,
        VAL_IMAGE_DIR / image_path.name
    )

    shutil.copy2(
        mask_path,
        VAL_MASK_DIR / mask_path.name
    )

# 確認 Validation 資料數量
val_images = list_images(VAL_IMAGE_DIR)
val_masks = list_images(VAL_MASK_DIR)

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Validation images:", len(val_images))
print("Validation masks:", len(val_masks))

# 防呆檢查
assert len(train_indices) == 4500
assert len(val_indices) == 500
assert len(val_images) == 500, "Validation 圖片數量不是 500"
assert len(val_masks) == 500, "Validation Mask 數量不是 500"

print("Training / Validation 切分完成")

Validation images: 500
Validation masks:  500
Remaining training samples: 4500


In [ ]:
# 確認GPU狀態
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GiB):",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        )
    )
else:
    print("目前沒有取得 GPU，先不要開始完整訓練。")

PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.56


In [ ]:
# 檢查資料尺寸、類別數與 Pixel Frequency
# NUM_CLASSES 與 EXPECTED_SIZE 皆由 Ground Truth 資料自動取得


#  掃描所有 Image / Mask 的尺寸 
image_sizes = set()
mask_sizes = set()

for img_path in train_images: #掃描所有training images
    with Image.open(img_path) as im: 
        image_sizes.add(im.size)   # PIL: (width, height)

for mask_path in train_masks: #掃描所有Ground Truth Masks
    with Image.open(mask_path) as mask:
        mask_sizes.add(mask.size)

print("Observed image sizes:", sorted(image_sizes)) #把實際測出的尺寸印出來
print("Observed mask sizes:", sorted(mask_sizes))

# 確認所有 image 尺寸一致
if len(image_sizes) != 1:
    raise ValueError(f"Image 尺寸不一致: {image_sizes}")

# 確認所有 mask 尺寸一致
if len(mask_sizes) != 1:
    raise ValueError(f"Mask 尺寸不一致: {mask_sizes}")

# 確認 image 與 mask 尺寸相同
if image_sizes != mask_sizes:
    raise ValueError(
        f"Image 與 Mask 尺寸不一致: "
        f"{image_sizes} vs {mask_sizes}"
    )

# 從實際資料取得影像尺寸
EXPECTED_SIZE = next(iter(image_sizes))


# ---------- 2. 掃描所有 Mask 中實際存在的 Label ID ----------
observed_labels = set() #建立一個空集合，用來記錄所有 Ground Truth Mask 中曾經出現過的 Label ID

for mask_path in train_masks:
    with Image.open(mask_path) as mask:
        arr = np.asarray(mask) #先用PIL開mask,再轉NumPy array

        # Semantic segmentation mask 應為單通道
        if arr.ndim != 2:
            raise ValueError(
                f"{mask_path.name} Mask 不是單通道: {arr.shape}"
            )

        observed_labels.update(np.unique(arr).tolist())

observed_labels = sorted(observed_labels) #把集合排序

# 確認 Label ID 從 0 開始且連續
expected_labels = list(range(max(observed_labels) + 1))

if observed_labels != expected_labels:
    raise ValueError(
        f"Label ID 不連續: {observed_labels}"
    )

# 從實際 Ground Truth Label IDs 得到類別數
NUM_CLASSES = len(observed_labels)


# ---------- 3. 統計各類別 Pixel Frequency ----------
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
errors = [] #建立空集合紀錄資料問題

for img_path in train_images: #在掃描所有training images

    mask_path = mask_by_stem[img_path.stem] #找出這張 image 對應的 Ground Truth Mask

    with Image.open(img_path) as im:
        if im.size != EXPECTED_SIZE: #驗證每一組image/mask並蒐集errors
            errors.append(
                (img_path.name, "image_size", im.size)
            )

    with Image.open(mask_path) as mask:

        arr = np.asarray(mask) #讀取mask並轉換numpy

        if mask.size != EXPECTED_SIZE:
            errors.append(
                (mask_path.name, "mask_size", mask.size)
            )

        if arr.ndim != 2: #再次確定mask是單通道二維
            errors.append(
                (mask_path.name, "mask_channels", arr.shape)
            )
            continue

        if arr.min() < 0 or arr.max() >= NUM_CLASSES: #檢查label範圍是0~23
            errors.append(
                (
                    mask_path.name,
                    "label_range",
                    int(arr.min()),
                    int(arr.max())
                )
            )
            continue

        label_counts += np.bincount( #把這張mask的統計加到整個dataset的累積結果
            arr.ravel(), #把二維攤平成一維
            minlength=NUM_CLASSES #確保輸出一定有24個位置，即使有些圖片裡沒有某些classes
        )[:NUM_CLASSES]


# ---------- 4. 顯示 Dataset 檢查結果 ----------
print("\n===== Dataset Information =====")
print("Expected size:", EXPECTED_SIZE)
print("Observed Label IDs:", observed_labels)
print("Number of classes:", NUM_CLASSES)

print("\n===== Dataset Check =====")
print("Errors:", len(errors))
print(errors[:10])

print(
    "Pixel frequency (%):",
    np.round(
        100 * label_counts / label_counts.sum(), #label_counts.sum() = 所有 pixels 總數
        4 #四捨五入到小數點後4位
    )
)

if errors:
    raise ValueError(
        "資料檢查未通過，先處理 errors。"
    )

Errors: 0
[]
Observed classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Pixel frequency (%): [5.71139e+01 5.17220e+00 2.38170e+01 2.68690e+00 6.17500e-01 1.83560e+00
 4.17500e-01 9.04900e-01 9.54400e-01 2.77530e+00 7.10100e-01 8.00000e-02
 6.81000e-01 8.66300e-01 2.80500e-01 1.97300e-01 1.50500e-01 9.26000e-02
 1.04800e-01 2.41600e-01 5.61000e-02 6.77000e-02 8.33000e-02 9.30000e-02]


In [ ]:
# 建立 PyTorch Semantic Segmentation Dataset

from pathlib import Path
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision import transforms

# ImageNet 預訓練 ResNet-50 使用的正規化參數
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    ),
])

class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = list(image_paths)
        self.mask_paths = list(mask_paths)

        if len(self.image_paths) != len(self.mask_paths):
            raise ValueError("Image 和 Mask 數量不一致")

        for image_path, mask_path in zip(
            self.image_paths, self.mask_paths
        ):
            if image_path.stem != mask_path.stem:
                raise ValueError(
                    f"Image / Mask 不匹配："
                    f"{image_path.name} vs {mask_path.name}"
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # 讀取 RGB 影像
        with Image.open(image_path) as im:
            image = im.convert("RGB")

        # 讀取單通道 Ground Truth Mask
        with Image.open(mask_path) as m:
            mask = np.asarray(m)

        if mask.ndim != 2:
            raise ValueError(
                f"Mask 不是單通道：{mask_path.name}"
            )

        if image.size != (mask.shape[1], mask.shape[0]):
            raise ValueError(
                f"Image / Mask 尺寸不一致：{image_path.name}"
            )

        if mask.min() < 0 or mask.max() >= NUM_CLASSES:
            raise ValueError(
                f"Mask Label 超出 0~{NUM_CLASSES - 1}："
                f"{mask_path.name}"
            )

        # Image：float tensor + ImageNet normalization
        image = image_transform(image)

        # Mask：保留原始類別 ID，不做 Normalize
        mask = torch.from_numpy(
            np.array(mask, dtype=np.int64, copy=True)
        ).long()

        return image, mask


# 建立完整 5000 張影像的 Dataset
# 後續使用 train_indices / val_indices 切分
train_dataset = SegmentationDataset(
    train_images,
    train_masks
)

assert len(train_dataset) == 5000

print("Dataset 建立成功")
print("Total samples:", len(train_dataset))
print("Number of classes:", NUM_CLASSES)

In [ ]:
# 初步Train
import csv
import random
import time
import numpy as np
import torch
import torch.nn as nn
import os

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50

# =========================
# 1. 設定
# =========================
SEED = 42
# NUM_CLASSES 已由前面的 Dataset Analysis 自動取得
BATCH_SIZE = 16
MAX_EPOCHS = 30
PATIENCE = 8

DEVICE = torch.device("cuda")
OUTPUT_DIR = Path("/content/drive/MyDrive/DL_HW1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#固定random seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 確認資料數量
assert len(train_images) == 5000
assert len(train_masks) == 5000
assert len(test_images) == 1000

# =========================
# 2. 固定切分訓練集與驗證集
# =========================
indices = list(range(len(train_images)))
random.Random(SEED).shuffle(indices)

val_indices = indices[:500] #前500筆當validation
train_indices = indices[500:] #後4500筆當training

train_subset = Subset(train_dataset, train_indices) #從完整train_dataset裡只取4500當training indices
val_subset = Subset(train_dataset, val_indices)

train_loader = DataLoader( #建立training data loader
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=os.cpu_count(),
    pin_memory=True,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False, #validation 不需要隨機打亂，只是要固定評估模型
    num_workers=os.cpu_count(),
    pin_memory=True,
)

print("Training samples:", len(train_subset))  
print("Validation samples:", len(val_subset))

# 儲存固定切分，後續實驗可以沿用
np.savez(
    OUTPUT_DIR / "split_seed42.npz",
    train_indices=np.array(train_indices),
    val_indices=np.array(val_indices),
)

# =========================
# 3. 建立模型
# =========================
# 使用 ImageNet 預訓練的 ResNet-50 backbone；
# DeepLabV3 的分割 head 則從頭訓練 24 個類別。
model = deeplabv3_resnet50(
    weights=None,
    weights_backbone=ResNet50_Weights.DEFAULT, #ResNet50 使用 ImageNet 預訓練權重
    num_classes=NUM_CLASSES,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW( #使用AdamW optimizer
    [
        {
            "params": model.backbone.parameters(),
            "lr": 1e-4,
        },
        {
            "params": model.classifier.parameters(),
            "lr": 5e-4,
        },
    ],
    weight_decay=1e-4, #加入weight decay，降低overfit
)

scaler = torch.amp.GradScaler("cuda")

# =========================
# 4. 計算固定 24 類別 mIoU
# =========================
@torch.no_grad()
def evaluate(model, loader): #定義Validation 評估函式
    model.eval()
    confusion = torch.zeros( #建立24*24的confusion matrix
        NUM_CLASSES, NUM_CLASSES,
        device=DEVICE, dtype=torch.int64
    )
    total_loss = 0.0 #累積整個 Validation Dataset 的 loss
    total_images = 0

    for images, masks in loader: #從 Validation DataLoader 一個 batch 一個 batch取資料
        images = images.to(DEVICE, non_blocking=True) #把 image 放到 GPU
        masks = masks.to(DEVICE, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images)["out"]
            loss = criterion(logits, masks) #計算 Validation Cross Entropy Loss
        preds = logits.argmax(dim=1) #logits轉class ID

        encoded = masks.reshape(-1) * NUM_CLASSES + preds.reshape(-1)
        confusion += torch.bincount( 
            encoded,
            minlength=NUM_CLASSES * NUM_CLASSES
        ).reshape(NUM_CLASSES, NUM_CLASSES)

        batch_size = images.size(0) #取得這個 batch 實際有幾張圖片
        total_loss += loss.item() * batch_size #累積 Validation loss，最後算整體平均
        total_images += batch_size

    tp = confusion.diag().float()
    gt_count = confusion.sum(dim=1).float() #每個 class 在 Ground Truth 中總共有多少 pixels
    pred_count = confusion.sum(dim=0).float() #模型總共預測多少 pixels 為該 class
    union = gt_count + pred_count - tp

    # 固定以 24 類別計算；union=0 的類別按 IoU=0 處理。
    iou = torch.where(
        union > 0,
        tp / union.clamp_min(1),
        torch.zeros_like(union),
    )

    miou = iou.mean().item()
    mean_loss = total_loss / total_images #計算Validation Dataset 的平均 loss

    return mean_loss, miou, iou.cpu().numpy()

# =========================
# 5. 訓練與驗證
# =========================
best_miou = -1.0
epochs_without_improvement = 0 #記錄已經連續幾個 epoch 沒有進步
history = []

for epoch in range(1, MAX_EPOCHS + 1): #起始值1，結束於 MAX_EPOCHS 之前
    start = time.time()
    model.train()

    running_loss = 0.0 #計算整個 epoch 的平均 Training Loss
    seen = 0

    for images, masks in train_loader: #4500張Training Data 一個 batch 一個 batch送進模型
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True) #每次新的 batch 開始前，把上一個 batch 的 gradient 清掉

        with torch.autocast("cuda", dtype=torch.float16): #pytorch自動混合精度(自動將數值降為16位元，節省GPU記憶體空間)
            logits = model(images)["out"] #圖片丟進模型算出預測結果
            loss = criterion(logits, masks) #損失函數比對模型預測和mask差多少，算出這一次誤差

        if not torch.isfinite(loss): #檢查算出來的loss是不是有限數字
            raise RuntimeError(
                f"Epoch {epoch} 出現非有限 loss: {loss.item()}"
            )

        scaler.scale(loss).backward() #loss放大再做反向傳播
        scaler.step(optimizer) #梯度縮放回來原本大小，再調整模型權重
        scaler.update() #進行下一輪前，動態調整放大倍數

        batch_size = images.size(0)#取得batch 數量
        running_loss += loss.item() * batch_size #累積 loss 和圖片數量
        seen += batch_size

    train_loss = running_loss / seen#得到這一個epoch的平均training loss
    val_loss, val_miou, class_iou = evaluate(model, val_loader)
    elapsed = time.time() - start

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_miou": val_miou,
        "seconds": elapsed,
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_mIoU={val_miou:.4f} | "
        f"time={elapsed / 60:.1f} min"
    )

    # 每個 epoch 都更新紀錄
    with open(OUTPUT_DIR / "training_log.csv", "w", newline="") as f: #儲存training log
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    if val_miou > best_miou: #如果現在的miou>best miou
        best_miou = val_miou #更新最佳成績
        epochs_without_improvement = 0 #early stopping 計數歸0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_miou": best_miou,
                "class_iou": class_iou,
                "seed": SEED,
                "num_classes": NUM_CLASSES,
            },
            OUTPUT_DIR / "best_deeplabv3_resnet50.pth",
        )

        print(f"  -> 已儲存最佳模型，val mIoU={best_miou:.4f}")
    else: #這次沒有進步，沒有進步的 epoch 數 加 1
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("驗證 mIoU 已連續多個 epoch 未改善，提前停止。")
        break

print("\n訓練完成")
print(f"Best validation mIoU: {best_miou:.4f}")
print("Checkpoint:", OUTPUT_DIR / "best_deeplabv3_resnet50.pth")
print("Training log:", OUTPUT_DIR / "training_log.csv")

In [ ]:
#第一次 fine-tune
# ============================================================
# DeepLabV3 + ResNet50 微調版（Google Colab）
# 會讀取 Epoch 11 的最佳 checkpoint，輸出到獨立資料夾，不覆寫原模型。
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import csv
import time
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

# -----------------------
# 1) 設定：先確認這些路徑
# -----------------------
DATA_ROOT = Path("/content/DL_HW1_Public")
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "image"
TRAIN_MASK_DIR  = DATA_ROOT / "train" / "gt"

# 原始最佳模型：只讀取，不覆寫
BASE_CHECKPOINT = Path(
    "/content/drive/MyDrive/DL_HW1_outputs/best_deeplabv3_resnet50.pth"
)

# 新實驗獨立輸出資料夾
OUTPUT_DIR = Path("/content/drive/MyDrive/DL_HW1_outputs/finetune_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 24
BATCH_SIZE = 16           
NUM_WORKERS = os.cpu_count()
FINETUNE_EPOCHS = 8
PATIENCE = 4             # 連續 4 輪沒有改善就提早停止
SEED = 42
IMAGE_SIZE = (640, 384)  # (width, height)
STOP_PATH = OUTPUT_DIR / "STOP"

# 這兩個 LR 是以原先 1e-4 / 5e-4 為前提，降低約 5 倍
BACKBONE_LR = 2e-5
CLASSIFIER_LR = 1e-4
WEIGHT_DECAY = 1e-4

# -----------------------
# 2) 基本檢查與可重現設定
# -----------------------
assert TRAIN_IMAGE_DIR.is_dir(), f"找不到影像資料夾：{TRAIN_IMAGE_DIR}"
assert TRAIN_MASK_DIR.is_dir(), f"找不到標註資料夾：{TRAIN_MASK_DIR}"
assert BASE_CHECKPOINT.is_file(), (
    f"找不到 checkpoint：{BASE_CHECKPOINT}\n"
    "請確認 Epoch 11 的最佳模型已存到 Google Drive。"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("警告：目前沒有使用 GPU，訓練可能會很慢。")

# 清掉上次留下的停止旗標，避免一開始就停止。
# 如果你是要停止「正在執行」的另一個訓練，請勿執行這個 cell。
if STOP_PATH.exists():
    STOP_PATH.unlink()

# -----------------------
# 3) 配對影像與 mask
# -----------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MASK_EXTS = {".png", ".bmp", ".tif", ".tiff"}

image_files = sorted( #找出所有符合格式的圖片
    p for p in TRAIN_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
mask_files = sorted( #找出所有符合格式mask
    p for p in TRAIN_MASK_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in MASK_EXTS
)
mask_by_stem = {p.stem: p for p in mask_files} #建立檔名對應字典
pairs = [(p, mask_by_stem[p.stem]) for p in image_files if p.stem in mask_by_stem]

if len(pairs) == 0: #如果配對成功的清單數量是0
    raise RuntimeError(
        "找不到任何影像/mask 配對。請檢查兩個資料夾中的檔名是否有相同 stem。"
    )
if len(pairs) != len(image_files): #如果配對成功的清單數量不等於圖片總數
    missing = [p.name for p in image_files if p.stem not in mask_by_stem][:10]
    raise RuntimeError(
        f"有 {len(image_files) - len(pairs)} 張影像找不到 mask。例：{missing}"
    )
print(f"配對成功：{len(pairs)} 張影像與 mask")

# -----------------------
# 4) Dataset / augmentation
#    mask 必須是 0..23 的整數標籤，不可做 Normalize
# -----------------------
imagenet_mean = (0.485, 0.456, 0.406) #定義imagenet資料集平均值
imagenet_std = (0.229, 0.224, 0.225)  #定義imagenet資料集標準差
image_to_tensor = transforms.Compose([ #建立影像轉換管線
    transforms.ToTensor(), #透過ToTensor()把圖片轉乘Pytorch數字張量
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std), #透過Normalize 減去平均值並除以標準差
])

#定義SegmentationDataset類別與初始化
class SegmentationDataset(Dataset): #繼承PytorchDataset類別，客製化資料讀取方式
    def __init__(self, image_mask_pairs, augment=False):
        self.pairs = image_mask_pairs
        self.augment = augment

    def __len__(self): #取得資料集大小
        return len(self.pairs)

    def __getitem__(self, idx): #讀取單筆資料
        image_path, mask_path = self.pairs[idx] #根據索引值拿出對應影像路徑和mask
        image = Image.open(image_path).convert("RGB") #使用PIL 圖片庫開啟影像（轉成RGB）
        mask = Image.open(mask_path) #開啟對應mask標註檔

       #確認mask是單通道格式，不能轉RGB(避免把原本代表類別編號的數值全部破壞掉)
        if mask.mode not in ("L", "P", "I", "I;16"):
            mask = mask.convert("L")

        if image.size != IMAGE_SIZE:#如果圖片尺寸不符合規定的大小，用雙線性插補進行平滑縮放
            image = image.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
        if mask.size != IMAGE_SIZE: #如果mask尺寸不符，用nearest進行縮放
            mask = mask.resize(IMAGE_SIZE, Image.Resampling.NEAREST)

        if self.augment and random.random() < 0.5: #augment=true且隨機抽樣機率小於0.5
            image = ImageOps.mirror(image) #影像做mirror
            mask = ImageOps.mirror(mask) #mask做mirror

        image = image_to_tensor(image) 
        mask_np = np.array(mask, dtype=np.int64)#mask轉numpy
        mask_tensor = torch.from_numpy(mask_np.copy()).long() #再轉Tensor(CrossEntropyLoss要求類別標籤必須是整數格式)

        min_label, max_label = int(mask_tensor.min()), int(mask_tensor.max())
        if min_label < 0 or max_label >= NUM_CLASSES: #檢查mask標籤數值有沒有符合合理數值0~23
            raise ValueError(
                f"mask 標籤超出 0..{NUM_CLASSES-1}：{mask_path.name}, "
                f"min={min_label}, max={max_label}"
            )
        return image, mask_tensor

# 固定 seed 切分。若原始訓練也使用同樣的 sorted 檔案順序與 seed=42，
# 此切分可重現；若原始 split 規則不同，需換成原本保存的 split。
indices = list(range(len(pairs)))
random.Random(SEED).shuffle(indices)

val_indices = indices[:500]
train_indices = indices[500:]

train_base = SegmentationDataset(pairs, augment=True) #開啟data augment
val_base = SegmentationDataset(pairs, augment=False) #關閉data augment(驗證時圖片要保持原貌)
train_set = Subset(train_base, train_indices)
val_set = Subset(val_base, val_indices)

train_loader = DataLoader( #建立data loader
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    drop_last=False
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    drop_last=False
)
print(f"Train={len(train_set)}, Validation={len(val_set)}")#印出這次訓練集和驗證集各有幾張圖片

# -----------------------
# 5) 建立模型並載入既有 checkpoint
# -----------------------
model = deeplabv3_resnet50( #建立空model
    weights=None, weights_backbone=None, num_classes=NUM_CLASSES
)

# 載入checkpoint。兼容常見儲存格式。
try:
    checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
except TypeError:  
    checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu")

if isinstance(checkpoint, dict): #自動解析checkpoint 內部結構
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        state_dict = checkpoint
else:
    raise TypeError("checkpoint 格式不是可辨識的 state_dict。")

# 兼容 DataParallel 存出的 module. 前綴
state_dict = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in state_dict.items()
    if isinstance(k, str)
}

missing, unexpected = model.load_state_dict(state_dict, strict=False) #權重塞進模型並檢查是否有遺漏
print("Checkpoint loaded.")
if missing:
    print("Missing keys:", missing[:12], "(total:", len(missing), ")")
if unexpected:
    print("Unexpected keys:", unexpected[:12], "(total:", len(unexpected), ")")

# 若 classifier 的 key/shape 不相容，不可默默接著訓練
bad_missing = [k for k in missing if k.startswith(("backbone.", "classifier."))]
bad_unexpected = [k for k in unexpected if k.startswith(("backbone.", "classifier."))]
if bad_missing or bad_unexpected:
    raise RuntimeError(
        "checkpoint 與目前 DeepLabV3 架構不一致，為避免錯誤微調已停止。\n"
        f"重要 missing keys={bad_missing[:10]}\n"
        f"重要 unexpected keys={bad_unexpected[:10]}\n"
        "請把這段錯誤貼回來，我會依 checkpoint 的實際架構修改。"
    )

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": BACKBONE_LR}, #套用小的LR
        {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR}, #套用較大的LR
    ],
    weight_decay=WEIGHT_DECAY, #加入權重衰減
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( #設定動態學習率調整器
    optimizer, mode="max", factor=0.5, patience=1
)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

# -----------------------
# 6) 評估：validation loss、24 類別 IoU、mIoU
# -----------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    confusion = torch.zeros(
        (NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=device
    )

    for images, masks in loader: #從驗證集的 loader 批次讀取圖片與標註，並送到 GPU
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")): #混和精度前向傳播
            logits = model(images)["out"]
            loss = criterion(logits, masks)

        bs = images.size(0)
        loss_sum += loss.item() * bs #累計loss
        sample_count += bs

        pred = logits.argmax(dim=1) #對每個 pixel 挑出 24 個類別分數最高的那一個，作為預測類別 ID
        valid = (masks >= 0) & (masks < NUM_CLASSES) #過濾掉不合法的標籤
        encoded = NUM_CLASSES * masks[valid].to(torch.int64) + pred[valid].to(torch.int64)
        confusion += torch.bincount(
            encoded, minlength=NUM_CLASSES * NUM_CLASSES
        ).reshape(NUM_CLASSES, NUM_CLASSES)
  
    #計算IoU、mIoU 與回傳結果
    confusion = confusion.cpu().numpy()
    intersection = np.diag(confusion).astype(np.float64)
    union = confusion.sum(axis=1) + confusion.sum(axis=0) - intersection
    iou = np.full(NUM_CLASSES, np.nan, dtype=np.float64)
    present = union > 0
    iou[present] = intersection[present] / union[present]
    miou = float(np.nanmean(iou)) if np.any(present) else 0.0
    val_loss = loss_sum / max(sample_count, 1)
    return val_loss, miou, iou, confusion

# 先評估載入的基準模型，避免用舊 log 的數字當作新 split 的基準
initial_val_loss, initial_miou, initial_iou, initial_confusion = evaluate(model, val_loader)
print(f"載入模型重新評估：val_loss={initial_val_loss:.4f}, val_mIoU={initial_miou:.4f}")
print("注意：若原訓練的 validation split 或前處理不同，這個數字可能與舊 log 不同。")

BEST_PATH = OUTPUT_DIR / "best_finetune_deeplabv3_resnet50.pth" #定義路徑
LATEST_PATH = OUTPUT_DIR / "latest_finetune_deeplabv3_resnet50.pth"
LOG_PATH = OUTPUT_DIR / "finetune_log.csv"
CLASS_IOU_PATH = OUTPUT_DIR / "per_class_iou.csv"
CONFUSION_PATH = OUTPUT_DIR / "confusion_matrix.csv"

best_miou = initial_miou
best_epoch = 0
epochs_without_improvement = 0

def save_checkpoint(path, epoch, val_loss, val_miou, val_iou): #定義儲存函數並存下初始備份
    torch.save({
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "best_val_miou": val_miou,
        "val_loss": val_loss,
        "num_classes": NUM_CLASSES,
        "backbone": "resnet50",
    }, path)

# 先把原始最佳權重複製到新的實驗目錄，確保即使第一輪中斷仍有可用模型
save_checkpoint(BEST_PATH, 0, initial_val_loss, initial_miou, initial_iou)
np.savetxt(CONFUSION_PATH, initial_confusion, delimiter=",", fmt="%d")
with open(CLASS_IOU_PATH, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "iou", "intersection_over_union_pixels"])
    for c in range(NUM_CLASSES):
        writer.writerow([c, "" if np.isnan(initial_iou[c]) else f"{initial_iou[c]:.6f}",
                         ""])

# -----------------------
# 7) 微調迴圈
# -----------------------
if not LOG_PATH.exists(): #初始化日誌檔案與路徑宣告
    with open(LOG_PATH, "w", newline="", encoding="utf-8-sig") as f:
        csv.writer(f).writerow([
            "finetune_epoch", "train_loss", "val_loss", "val_miou",
            "backbone_lr", "classifier_lr", "epoch_seconds", "best_val_miou"
        ])

print(f"新實驗輸出：{OUTPUT_DIR}")
print(f"最佳模型：{BEST_PATH}")

#Training 階段
for epoch in range(1, FINETUNE_EPOCHS + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    seen = 0

    for step, (images, masks) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)["out"]
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = images.size(0)
        running_loss += loss.item() * bs
        seen += bs

        if step % 100 == 0:
            print(f"Epoch {epoch:02d}/{FINETUNE_EPOCHS} "
                  f"step {step}/{len(train_loader)} loss={loss.item():.4f}")

    train_loss = running_loss / max(seen, 1)
    val_loss, val_miou, val_iou, confusion = evaluate(model, val_loader) #Validation 階段
    scheduler.step(val_miou)

    improved = val_miou > best_miou #成績評比與「最佳模型」存檔
    if improved:
        best_miou = val_miou
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(BEST_PATH, epoch, val_loss, val_miou, val_iou)
        np.savetxt(CONFUSION_PATH, confusion, delimiter=",", fmt="%d")
        with open(CLASS_IOU_PATH, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["class_id", "iou"])
            for c in range(NUM_CLASSES):
                writer.writerow([c, "" if np.isnan(val_iou[c]) else f"{val_iou[c]:.6f}"])
    else:
        epochs_without_improvement += 1

    save_checkpoint(LATEST_PATH, epoch, val_loss, val_miou, val_iou)

    #記錄成績、印出狀態與防呆中斷機制
    elapsed = time.time() - start_time
    backbone_lr_now = optimizer.param_groups[0]["lr"]
    classifier_lr_now = optimizer.param_groups[1]["lr"]
    with open(LOG_PATH, "a", newline="", encoding="utf-8-sig") as f:
        csv.writer(f).writerow([
            epoch, f"{train_loss:.6f}", f"{val_loss:.6f}", f"{val_miou:.6f}",
            f"{backbone_lr_now:.8g}", f"{classifier_lr_now:.8g}",
            f"{elapsed:.1f}", f"{best_miou:.6f}"
        ])

    print(
        f"Fine-tune Epoch {epoch:02d}/{FINETUNE_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_mIoU={val_miou:.4f} | best={best_miou:.4f} | "
        f"time={elapsed/60:.1f} min"
        + (" -> 儲存最佳模型" if improved else "")
    )

    # 安全停止：完成一個 epoch、寫好 checkpoint 和 log 後才停止
    if STOP_PATH.exists():
        print("偵測到 STOP 檔案：已完成目前 epoch 並安全停止。")
        break

    if epochs_without_improvement >= PATIENCE:
        print(f"連續 {PATIENCE} 個 epoch 未改善，提早停止。")
        break

print("\n微調完成。")
print(f"最佳 mIoU：{best_miou:.4f}（本次微調 epoch={best_epoch}；0 表示載入模型的初始評估最佳）")
print(f"最佳 checkpoint：{BEST_PATH}")
print(f"訓練紀錄：{LOG_PATH}")
print(f"24 類別 IoU：{CLASS_IOU_PATH}")
print(f"混淆矩陣：{CONFUSION_PATH}")
print("原始 checkpoint 未被覆寫。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
配對成功：5000 張影像與 mask
Train=4500, Validation=500
Checkpoint loaded.


/tmp/ipykernel_1031/3110543410.py:247: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_1031/3110543410.py:265: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


載入模型重新評估：val_loss=0.5314, val_mIoU=0.4015
注意：若原訓練的 validation split 或前處理不同，這個數字可能與舊 log 不同。
新實驗輸出：/content/drive/MyDrive/DL_HW1_outputs/finetune_v1
最佳模型：/content/drive/MyDrive/DL_HW1_outputs/finetune_v1/best_finetune_deeplabv3_resnet50.pth


/tmp/ipykernel_1031/3110543410.py:349: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch 01/8 step 100/282 loss=0.1760
Epoch 01/8 step 200/282 loss=0.1481
Fine-tune Epoch 01/8 | train_loss=0.1498 | val_loss=0.4913 | val_mIoU=0.4132 | best=0.4132 | time=6.3 min -> 儲存最佳模型
Epoch 02/8 step 100/282 loss=0.1227
Epoch 02/8 step 200/282 loss=0.1180
Fine-tune Epoch 02/8 | train_loss=0.1328 | val_loss=0.5011 | val_mIoU=0.4156 | best=0.4156 | time=6.3 min -> 儲存最佳模型
Epoch 03/8 step 100/282 loss=0.1343
Epoch 03/8 step 200/282 loss=0.1237
Fine-tune Epoch 03/8 | train_loss=0.1244 | val_loss=0.4987 | val_mIoU=0.4156 | best=0.4156 | time=6.4 min -> 儲存最佳模型
Epoch 04/8 step 100/282 loss=0.1232
Epoch 04/8 step 200/282 loss=0.1445
Fine-tune Epoch 04/8 | train_loss=0.1196 | val_loss=0.5133 | val_mIoU=0.4133 | best=0.4156 | time=6.3 min
Epoch 05/8 step 100/282 loss=0.1437
Epoch 05/8 step 200/282 loss=0.0930
Fine-tune Epoch 05/8 | train_loss=0.1138 | val_loss=0.5156 | val_mIoU=0.4151 | best=0.4156 | time=6.3 min
Epoch 06/8 step 100/282 loss=0.1146
Epoch 06/8 step 200/282 loss=0.1127
Fine-tun

In [ ]:
#第二次 fine-tune
import csv
import random
import time
from pathlib import Path
from contextlib import nullcontext

import numpy as np
from PIL import Image
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms import functional as TF
from torchvision.models.segmentation import deeplabv3_resnet50


# ==================================================
# 1. 設定
# ==================================================
SEED = 42
NUM_CLASSES = 24

BATCH_SIZE = 16
MAX_EPOCHS = 8
PATIENCE = 4

BACKBONE_LR = 1e-5
CLASSIFIER_LR = 5e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = os.cpu_count()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BASE_DIR = Path("/content/drive/MyDrive/DL_HW1_outputs")

CHECKPOINT_PATH = (
    BASE_DIR / "finetune_v1" /
    "best_finetune_deeplabv3_resnet50.pth"
)

SPLIT_PATH = BASE_DIR / "split_seed42.npz"

OUT_DIR = BASE_DIR / "finetune_weighted_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/content/DL_HW1_Public")
IMAGE_DIR = DATA_ROOT / "train" / "image"
MASK_DIR = DATA_ROOT / "train" / "gt"

random.seed(SEED) #確認實驗可重現性
np.random.seed(SEED)
torch.manual_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Output:", OUT_DIR)


# ==================================================
# 2. 檢查輸入檔案
# ==================================================
assert CHECKPOINT_PATH.is_file(), (
    f"找不到最佳模型：{CHECKPOINT_PATH}"
)

assert SPLIT_PATH.is_file(), (
    f"找不到固定切分：{SPLIT_PATH}"
)

assert IMAGE_DIR.is_dir(), f"找不到影像資料夾：{IMAGE_DIR}"
assert MASK_DIR.is_dir(), f"找不到 mask 資料夾：{MASK_DIR}"


# ==================================================
# 3. 建立影像與 mask 配對
# ==================================================
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

image_files = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONS
)

mask_files = sorted(
    p for p in MASK_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONS
)

mask_by_stem = {p.stem: p for p in mask_files}

pairs = [
    (img, mask_by_stem[img.stem])
    for img in image_files
    if img.stem in mask_by_stem
]

if len(pairs) != 5000:
    raise RuntimeError(
        f"只配對到 {len(pairs)} 組，預期 5000 組。"
        "請檢查影像與 mask 的檔名。"
    )

print("配對成功：", len(pairs))


# ==================================================
# 4. 定義Dataset類別和資料前處理
# ==================================================
# 與原本模型一致：RGB + ImageNet normalization。
# 訓練時只做隨機水平翻轉，驗證時不做隨機增強。

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class SegmentationDataset(Dataset):
    def __init__(self, pairs, train=False):
        self.pairs = pairs
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx): #讀取與安全檢查
        image_path, mask_path = self.pairs[idx]

        with Image.open(image_path) as im:
            image = im.convert("RGB")

        with Image.open(mask_path) as m:
            mask = np.array(m, dtype=np.int64)

        if mask.ndim != 2:
            raise ValueError(
                f"mask 必須是單通道標籤圖：{mask_path}, "
                f"shape={mask.shape}"
            )

        if mask.min() < 0 or mask.max() >= NUM_CLASSES:
            raise ValueError(
                f"標籤超出 0..23：{mask_path}, "
                f"min={mask.min()}, max={mask.max()}"
            )

        mask = torch.from_numpy(mask.copy())
        image = TF.to_tensor(image)
        image = TF.normalize(image, MEAN, STD)

        if self.train and random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        return image, mask


# ==================================================
# 5. 載入原始固定切分
# ==================================================
split = np.load(SPLIT_PATH)

train_indices = split["train_indices"].astype(int).tolist()
val_indices = split["val_indices"].astype(int).tolist()

if len(train_indices) != 4500 or len(val_indices) != 500:
    raise RuntimeError(
        f"切分數量錯誤：train={len(train_indices)}, "
        f"val={len(val_indices)}"
    )

if sorted(train_indices + val_indices) != list(range(len(pairs))):
    raise RuntimeError(
        "切分索引有重複、遺漏或超出範圍。"
    )

# 重要：split 中的索引必須對應原始 train_dataset 的順序。
# 這裡使用排序後的 pairs；請確認原始資料集也是這個順序。

train_dataset = SegmentationDataset(pairs, train=True)
val_dataset = SegmentationDataset(pairs, train=False)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"Train={len(train_subset)}, Validation={len(val_subset)}")


# ==================================================
# 6. 計算溫和類別權重
# ==================================================
# 使用訓練集的像素分布。
# 權重與像素頻率的平方根成反比，不使用極端反頻率。
# 最後限制權重並正規化，使平均權重約為 1。

pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

print("正在統計訓練集類別像素數量...")

for n, idx in enumerate(train_indices, start=1):
    _, mask_path = pairs[idx]

    with Image.open(mask_path) as m:
        mask = np.array(m, dtype=np.int64)

    if mask.ndim != 2:
        raise ValueError(f"mask 不是單通道：{mask_path}")

    if mask.min() < 0 or mask.max() >= NUM_CLASSES:
        raise ValueError(f"mask label 超出範圍：{mask_path}")

    pixel_counts += np.bincount(
        mask.ravel(), minlength=NUM_CLASSES
    )

    if n % 1000 == 0:
        print(f"已統計 {n}/{len(train_indices)} 張")

if np.any(pixel_counts == 0): #防呆檢查
    missing = np.where(pixel_counts == 0)[0].tolist()
    raise RuntimeError(
        f"訓練集中有類別沒有任何像素：{missing}"
    )

freq = pixel_counts / pixel_counts.sum()

weights = 1.0 / np.sqrt(freq)
weights = weights / weights.mean()

# 限制稀有類別的權重，避免過度放大
weights = np.clip(weights, 0.5, 3.0)
weights = weights / weights.mean()

class_weights = torch.tensor(
    weights, dtype=torch.float32, device=DEVICE
)

with open(OUT_DIR / "class_weights.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "class_id", "pixel_count", "frequency", "weight"
    ])

    for cid in range(NUM_CLASSES):
        writer.writerow([
            cid,
            int(pixel_counts[cid]),
            float(freq[cid]),
            float(weights[cid]),
        ])

print("類別權重：")
print(np.round(weights, 3).tolist())


# ==================================================
# 7. 載入上一輪最佳模型
# ==================================================
model = deeplabv3_resnet50(
    weights=None,
    weights_backbone=None,
    num_classes=NUM_CLASSES,
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

state_dict = checkpoint.get(
    "model_state_dict", checkpoint
)

model.load_state_dict(state_dict, strict=True)

print("Checkpoint loaded.")
print(
    "上一輪最佳 mIoU：",
    checkpoint.get("best_val_miou", "checkpoint 未記錄")
)


# ==================================================
# 8. Loss 與 optimizer
# ==================================================
# 訓練使用加權 CE。
# 驗證使用一般 CE，方便和上一輪 val_loss 比較。
train_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

val_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.backbone.parameters(),
            "lr": BACKBONE_LR,
        },
        {
            "params": model.classifier.parameters(),
            "lr": CLASSIFIER_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
)

scaler = torch.amp.GradScaler(
    "cuda", enabled=(DEVICE.type == "cuda")
)


# ==================================================
# 9. 驗證函數：固定 24 類別 mIoU
# ==================================================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    confusion = torch.zeros(
        NUM_CLASSES,
        NUM_CLASSES,
        device=DEVICE,
        dtype=torch.int64,
    )

    total_loss = 0.0
    total_images = 0

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        amp_context = (
            torch.amp.autocast("cuda")
            if DEVICE.type == "cuda"
            else nullcontext()
        )

        with amp_context:
            logits = model(images)["out"]
            loss = val_criterion(logits, masks)

        preds = logits.argmax(dim=1)

        encoded = (
            masks.reshape(-1) * NUM_CLASSES
            + preds.reshape(-1)
        )

        confusion += torch.bincount(
            encoded,
            minlength=NUM_CLASSES * NUM_CLASSES,
        ).reshape(NUM_CLASSES, NUM_CLASSES)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    tp = confusion.diag().float()
    gt_count = confusion.sum(dim=1).float()
    pred_count = confusion.sum(dim=0).float()
    union = gt_count + pred_count - tp

    iou = torch.where(
        union > 0,
        tp / union.clamp_min(1),
        torch.zeros_like(union),
    )

    miou = iou.mean().item()
    mean_loss = total_loss / max(total_images, 1)

    return (
        mean_loss,
        miou,
        iou.cpu().numpy(),
        confusion.cpu().numpy(),
    )


# ==================================================
# 10. 先評估初始模型
# ==================================================
initial_loss, initial_miou, initial_iou, initial_cm = evaluate(
    model, val_loader
)

print(
    f"微調前：val_loss={initial_loss:.4f}, "
    f"val_mIoU={initial_miou:.4f}"
)

# 上一輪最佳值是 0.4190；若這裡差距明顯，停止檢查資料順序
# 前處理與驗證切分，不直接開始加權訓練
if abs(initial_miou - 0.4190) > 0.015:
    raise RuntimeError(
        f"初始 mIoU={initial_miou:.4f}，與預期 0.4190 差距較大。"
        "為避免在不一致的驗證設定上做實驗，程式已停止。"
    )


# ==================================================
# 11. 儲存初始模型，並準備訓練紀錄
# ==================================================
best_miou = initial_miou
best_epoch = 0
no_improvement = 0

def save_best(path, epoch, miou, class_iou, cm):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_miou": miou,
            "class_iou": class_iou,
            "seed": SEED,
            "num_classes": NUM_CLASSES,
            "source_checkpoint": str(CHECKPOINT_PATH),
            "class_weights": weights.tolist(),
        },
        path,
    )

    with open(OUT_DIR / "per_class_iou.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_id", "iou"])
        writer.writerows([
            [i, float(v)] for i, v in enumerate(class_iou)
        ])

    np.savetxt(
        OUT_DIR / "confusion_matrix.csv",
        cm,
        delimiter=",",
        fmt="%d",
    )


BEST_PATH = OUT_DIR / "best_weighted_deeplabv3_resnet50.pth"

save_best(
    BEST_PATH, 0, best_miou, initial_iou, initial_cm
)

history = [{
    "epoch": 0,
    "train_loss_weighted": "",
    "val_loss": initial_loss,
    "val_miou": initial_miou,
    "backbone_lr": BACKBONE_LR,
    "classifier_lr": CLASSIFIER_LR,
    "seconds": 0.0,
}]


# ==================================================
# 12. 開始加權微調
# ==================================================
for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()
    model.train()

    running_loss = 0.0
    seen = 0

    for step, (images, masks) in enumerate(
        train_loader, start=1
    ):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        amp_context = (
            torch.amp.autocast("cuda")
            if DEVICE.type == "cuda"
            else nullcontext()
        )

        with amp_context:
            logits = model(images)["out"]
            loss = train_criterion(logits, masks)

        if not torch.isfinite(loss): #數值防爆與反向傳播
            raise RuntimeError(
                f"Epoch {epoch} loss 非有限值：{loss.item()}"
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        seen += batch_size

        if step % 100 == 0:
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS} "
                f"step {step}/{len(train_loader)} "
                f"weighted_loss={loss.item():.4f}"
            )

    train_loss = running_loss / max(seen, 1)

    val_loss, val_miou, class_iou, cm = evaluate(#驗證考試、學習率調整與最佳模型儲存
        model, val_loader
    )

    elapsed = time.time() - start

    improved = val_miou > best_miou

    if improved:
        best_miou = val_miou
        best_epoch = epoch
        no_improvement = 0

        save_best(
            BEST_PATH,
            epoch,
            best_miou,
            class_iou,
            cm,
        )
    else:
        no_improvement += 1

    scheduler.step(val_miou)

    record = { #寫入日誌、印出總結與Early Stopping
        "epoch": epoch,
        "train_loss_weighted": train_loss,
        "val_loss": val_loss,
        "val_miou": val_miou,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "classifier_lr": optimizer.param_groups[1]["lr"],
        "seconds": elapsed,
    }
    history.append(record)

    with open(
        OUT_DIR / "weighted_finetune_log.csv",
        "w",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=history[0].keys()
        )
        writer.writeheader()
        writer.writerows(history)

    print(
        f"Weighted Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_mIoU={val_miou:.4f} | "
        f"best={best_miou:.4f} | "
        f"time={elapsed / 60:.1f} min"
        + (" -> 儲存最佳模型" if improved else "")
    )

    if no_improvement >= PATIENCE:
        print("驗證 mIoU 連續未改善，提前停止。")
        break


# ==================================================
# 13. 完成
# ==================================================
print("\n類別加權微調完成")
print(f"微調前 mIoU：{initial_miou:.4f}")
print(f"最佳 mIoU：{best_miou:.4f}")
print(f"最佳 epoch：{best_epoch}")
print("最佳模型：", BEST_PATH)
print("訓練紀錄：", OUT_DIR / "weighted_finetune_log.csv")
print("類別權重：", OUT_DIR / "class_weights.csv")
print("類別 IoU：", OUT_DIR / "per_class_iou.csv")
print("混淆矩陣：", OUT_DIR / "confusion_matrix.csv")
print("原始模型與上一輪模型未被覆寫。")

Device: cuda
Checkpoint: /content/drive/MyDrive/DL_HW1_outputs/finetune_v1/best_finetune_deeplabv3_resnet50.pth
Output: /content/drive/MyDrive/DL_HW1_outputs/finetune_weighted_v1
配對成功： 5000
Train=4500, Validation=500
正在統計訓練集類別像素數量...
已統計 1000/4500 張
已統計 2000/4500 張
已統計 3000/4500 張
已統計 4000/4500 張
類別權重：
[0.469, 0.469, 0.469, 0.469, 0.635, 0.469, 0.762, 0.514, 0.514, 0.469, 0.597, 1.91, 0.599, 0.532, 0.986, 1.103, 1.28, 1.623, 1.533, 0.998, 2.259, 2.049, 1.647, 1.641]
Checkpoint loaded.
上一輪最佳 mIoU： 0.4190171234880524
微調前：val_loss=0.5156, val_mIoU=0.4190
Epoch 01/8 step 100/282 weighted_loss=0.1417
Epoch 01/8 step 200/282 weighted_loss=0.1114
Weighted Epoch 01/8 | train_loss=0.1071 | val_loss=0.5364 | val_mIoU=0.4195 | best=0.4195 | time=6.1 min -> 儲存最佳模型
Epoch 02/8 step 100/282 weighted_loss=0.0995
Epoch 02/8 step 200/282 weighted_loss=0.0971
Weighted Epoch 02/8 | train_loss=0.1035 | val_loss=0.5439 | val_mIoU=0.4175 | best=0.4195 | time=6.1 min
Epoch 03/8 step 100/282 weighted_loss=0.10